# Guarded hierarchical residual challenger

This research-only notebook attempts to retain the aggregate seasonal/seascape gain without sacrificing individual sources, eras, or regions. It keeps the constrained current/transport blend as a probability offset and learns only a bounded residual correction. Production models, canonical Parquet files, and release pointers are never modified.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if not (HERE / 'guarded_residual_experiment.py').is_file():
    raise RuntimeError('Run this notebook from notebooks/cetaceans/killer_whales/imputation/testing')
import os
REPO_ROOT = Path(os.environ["MARINE_MAMMALS_WORKSPACE_ROOT"]).expanduser().resolve()
for path in (HERE,):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from experiment_support import resolve_release_paths
from guarded_residual_experiment import run_guarded_residual_experiment

paths = resolve_release_paths(REPO_ROOT)
output_dir = Path(os.environ['MARINE_MAMMALS_RESEARCH_OUTPUT_ROOT']).expanduser().resolve() / 'guarded_residual'
paths

## `P_OTHER` is not abstention

`P_OTHER` represents biological probability that a supported observation belongs outside the modeled SRKW/Transient set. Abstention represents insufficient model support or reliability. A supported Other prediction contributes expected Other mass. An abstained row contributes unit expected Unknown mass, while any class scores remain diagnostic only.

This notebook improves only the conditional SRKW-versus-Transient component. It does not emit or infer `P_OTHER`.

## Model design

1. Rebuild leakage-controlled marine transport support inside each outer fold.
2. Construct the source-safe constrained transport blend.
3. Treat its log-odds as an offset and learn a capped residual from 16 seasonal variables, a reduced physical seascape panel, and selected seasonal interactions.
4. Fit regularized source/era/region/evidence/coverage calibration using cross-fitted residual scores.
5. Estimate fold-local reliability weights and shrink unreliable rows toward the safe offset baseline.

In [ ]:
experiment = run_guarded_residual_experiment(output_dir, paths)
comparison = experiment['guarded_residual_comparison'].sort_values(['log_loss', 'brier'])
comparison

## Guardrails against the safe transport baseline

Required source, era, and H3-R4 strata contain at least 100 independent encounters and may not degrade Brier by more than 10%. A recommendation must also improve aggregate Brier and log loss, have ECE at most 0.03, calibration intercept within ±0.10, and calibration slope between 0.8 and 1.2.

In [ ]:
experiment['guarded_residual_gate_summary'].sort_values(['model', 'stratification'])

## Adaptive reliability weights

A challenger weight of zero is an explicit fallback to the safe transport probability. Larger values admit more of the calibrated residual prediction. All weights are learned from outer-training OOF predictions and then applied to the untouched outer test fold.

In [ ]:
experiment['guarded_residual_fold_selections']

## Stable residual signals

Coefficients are standardized residual effects. They describe association after the safe transport offset and should not be interpreted causally.

In [ ]:
(experiment['guarded_residual_top_coefficients']
 .groupby('feature', as_index=False)
 .agg(mean_abs_coefficient=('absolute_coefficient', 'mean'), folds_selected=('outer_fold', 'nunique'))
 .sort_values(['folds_selected', 'mean_abs_coefficient'], ascending=[False, False])
 .head(25))

In [ ]:
print('Recommended research challenger:', experiment['recommended_research_challenger'])
print('Manifest:', experiment['manifest_path'])
print('Production promotion eligible: False')